# v_calib_xy2 Full Pipeline (Standalone)

모든 모델/데이터셋 코드를 인라인으로 포함한 제출용 노트북.

```
1. 5-Fold 학습  (train_parallel.py -v v_calib_xy2 --seed 42 와 동일)
2. Optuna 윈도우 가중치 최적화 + 테스트 추론  (predict_optimized.py -v v_calib_xy2 --trial 1000 와 동일)
```

## 0. 설정

In [1]:
import os, sys, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import StepLR
from sklearn.model_selection import KFold
import optuna

# 노트북이 note/ 아래에 있으므로 프로젝트 루트로 이동
PROJ_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(PROJ_ROOT)
print(f"Project root: {PROJ_ROOT}")

# ── 학습 설정 ──────────────────────────────────────────────
VERSION         = "v_calib_xy2"
SEED            = 42
EPOCHS          = 50
BATCH_SIZE      = 1024
LR_DECAY_STEP   = 8
LR_DECAY_GAMMA  = 0.5
PAD_MODE        = "acc"
MIN_WIN         = 3
SLIDING_MODE    = "extended"

# ── 추론 설정 ──────────────────────────────────────────────
TRIALS      = 1000   # Optuna trials
TTA_ANGLES  = 1
INFER_BATCH = 4096

# ── 경로 설정 ──────────────────────────────────────────────
CKPT_DIR    = "checkpoints"
CACHE_DIR   = "data/cache"
TEST_DIR    = "data/test"
SAMPLE_SUB  = "data/sample_submission.csv"

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Device: {device}")

os.makedirs(CKPT_DIR, exist_ok=True)

Project root: /Users/leejongmin/code/mosquito
Device: mps


## 1. 유틸리티 함수 (model/utils.py)

In [2]:
def _ema_va_local(diffs_local, alpha, beta):
    B, T, _ = diffs_local.shape
    one_m_a = 1.0 - alpha
    one_m_b = 1.0 - beta
    vs = diffs_local.new_empty(B, T, 3)
    v = diffs_local[:, 0]
    vs[:, 0] = v
    for t in range(1, T):
        v = alpha * diffs_local[:, t] + one_m_a * v
        vs[:, t] = v
    vl = vs[:, -1]
    ad = vs[:, 1:] - vs[:, :-1]
    a = ad[:, 0]
    for t in range(1, T - 1):
        a = beta * ad[:, t] + one_m_b * a
    return vl, a


def _soft_hit_loss(pred, target, thr, k):
    return (1 - torch.sigmoid(-(torch.norm(pred - target, dim=1) - thr) * k)).mean()


def extract_features(X, mean_stats=None, std_stats=None, dir_net=None, heading_mode="3step"):
    device = X.device
    p_last = X[:, 10]
    diffs  = X[:, 1:] - X[:, :-1]  # (B, 10, 3)

    n1 = diffs[:, 1:].norm(dim=2, keepdim=True) + 1e-8
    n2 = diffs[:, :-1].norm(dim=2, keepdim=True) + 1e-8
    cos_t     = ((diffs[:,1:]*diffs[:,:-1]).sum(2,keepdim=True)/(n1*n2)).clamp(-1,1)
    theta_seq = torch.acos(cos_t).squeeze(2)  # (B,9)

    theta       = theta_seq[:, -1:]
    theta_mean  = theta_seq.mean(1, keepdim=True)
    theta_std   = theta_seq.std(1,  keepdim=True)
    theta_vel   = theta_seq[:,-1:] - theta_seq[:,-2:-1]
    theta_acc   = theta_seq[:,-1:] - 2*theta_seq[:,-2:-1] + theta_seq[:,-3:-2]
    theta_trend = theta_seq[:,-1:] - theta_seq[:,-3:].mean(1,keepdim=True)

    if dir_net is not None:
        speed_seq = diffs.norm(dim=2)
        state = torch.cat([speed_seq, theta_seq], dim=1)
        if dir_net[0].in_features == 29:
            z_speed_seq = diffs[:, :, 2].abs()
            state = torch.cat([state, z_speed_seq], dim=1)
        weights = F.softmax(dir_net(state), dim=1)
        v_sm = (diffs * weights.unsqueeze(2)).sum(dim=1)
    else:
        if heading_mode == "3step":
            v_sm = (3*diffs[:,-1] + 2*diffs[:,-2] + diffs[:,-3]) / 6.0
        elif heading_mode == "1step":
            v_sm = diffs[:, -1]
        elif heading_mode == "2step":
            v_sm = (2*diffs[:,-1] + diffs[:,-2]) / 3.0
        elif heading_mode == "4step":
            v_sm = (4*diffs[:,-1]+3*diffs[:,-2]+2*diffs[:,-3]+diffs[:,-4]) / 10.0
        elif heading_mode == "5step":
            v_sm = (5*diffs[:,-1]+4*diffs[:,-2]+3*diffs[:,-3]+2*diffs[:,-4]+diffs[:,-5]) / 15.0
        else:
            v_sm = diffs.mean(dim=1)

    fwd   = v_sm / (v_sm.norm(dim=1,keepdim=True) + 1e-8)
    up_w  = torch.zeros_like(fwd); up_w[:,2] = 1.0
    up_w[fwd[:,2].abs()>0.99] = torch.tensor([0.,1.,0.], device=device)
    right = torch.cross(fwd, up_w, dim=1)
    right = right / (right.norm(dim=1,keepdim=True)+1e-8)
    up    = torch.cross(right, fwd, dim=1)
    up    = up / (up.norm(dim=1,keepdim=True)+1e-8)
    R     = torch.stack([fwd, right, up], dim=2)  # (B,3,3)

    v_last  = diffs[:,-1]
    v_prev1 = diffs[:,-2]
    speed   = v_last.norm(dim=1, keepdim=True)
    a_last  = v_last - v_prev1
    acc_mag = a_last.norm(dim=1, keepdim=True)
    v_local = torch.matmul(v_last.unsqueeze(1), R).squeeze(1)
    a_local = torch.matmul(a_last.unsqueeze(1), R).squeeze(1)

    X_local     = torch.matmul(X - p_last.unsqueeze(1), R)
    p_std_local = X_local.std(1)
    v_local_abs = v_local.abs()
    jerk_g      = diffs[:,-1] - 2*diffs[:,-2] + diffs[:,-3]
    jerk_l      = torch.matmul(jerk_g.unsqueeze(1), R).squeeze(1)
    jerk_mag    = jerk_g.norm(dim=1, keepdim=True)

    features = torch.cat([
        v_local, a_local, speed, acc_mag,
        theta, theta_mean, theta_std, theta_trend, theta_vel, theta_acc,
        p_std_local, v_local_abs, jerk_l, jerk_mag
    ], dim=1)  # (B, 24)

    if mean_stats is None or std_stats is None:
        mean_stats = features.mean(0, keepdim=True)
        std_stats  = features.std(0,  keepdim=True) + 1e-8
    features_scaled = (features - mean_stats) / std_stats

    return features_scaled, diffs, p_last, theta, theta_mean, theta_std, theta_seq, R, speed, mean_stats, std_stats


def rodrigues_rotate(v, omega):
    angle = omega.norm(dim=1, keepdim=True)
    axis  = omega / (angle + 1e-8)
    cos_a = torch.cos(angle)
    sin_a = torch.sin(angle)
    cross = torch.cross(axis, v, dim=1)
    dot   = (axis * v).sum(dim=1, keepdim=True)
    return v * cos_a + cross * sin_a + axis * dot * (1.0 - cos_a)


class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(dim, dim)
        )
        self.ln = nn.LayerNorm(dim)
    def forward(self, x):
        return self.ln(x + self.net(x))


class BaseHyperPhysicsModel(nn.Module):
    def __init__(self, sh_thr=0.010758, sh_k=395.4, huber_delta=0.0105, huber_w=60.557):
        super().__init__()
        self.sh_thr      = sh_thr
        self.sh_k        = sh_k
        self.huber_delta = huber_delta
        self.huber_w     = huber_w
        self.lr          = 1e-2
        self.wd          = 3e-3

    def get_param_groups(self, lr, weight_decay):
        decay, no_decay = [], []
        for name, param in self.named_parameters():
            if not param.requires_grad:
                continue
            if len(param.shape) == 1 or name.endswith(".bias"):
                no_decay.append(param)
            else:
                decay.append(param)
        return [
            {"params": decay,    "lr": lr, "weight_decay": weight_decay},
            {"params": no_decay, "lr": lr, "weight_decay": 0.0}
        ]

    def compute_loss(self, pp, yr, pred_local=None, yr_local=None, log_var=None, p_last=None, theta=None):
        sh = _soft_hit_loss(pp, yr, thr=self.sh_thr, k=self.sh_k)
        h  = F.huber_loss(pp, yr, delta=self.huber_delta)
        return sh + self.huber_w * h

    def get_features(self, X, mean_stats=None, std_stats=None):
        dir_net      = getattr(self, "dir_net", None)
        heading_mode = getattr(self, "heading_mode", "3step")
        return extract_features(X, mean_stats, std_stats, dir_net, heading_mode)

    def predict_batch_tta(self, coords, mean, std, n_angles=16, use_mirrors=False):
        if n_angles <= 1:
            ft, df, plt_, tht, _, _, _, Rt, spt, _, _ = self.get_features(coords, mean, std)
            out = self.forward(ft, df, plt_, tht, spt, Rt)
            return out[0] if isinstance(out, tuple) else out

        dev = coords.device
        B   = coords.shape[0]
        angs = torch.linspace(0, 2 * np.pi, n_angles + 1, device=dev)[:-1]
        pl = coords[:, 10]
        coords_centered = coords - pl.unsqueeze(1)

        c = torch.cos(angs); s = torch.sin(angs)
        zeros = torch.zeros_like(c); ones = torch.ones_like(c)
        Rz_t = torch.stack([
            torch.stack([c, s, zeros], dim=-1),
            torch.stack([-s, c, zeros], dim=-1),
            torch.stack([zeros, zeros, ones], dim=-1)
        ], dim=-2)
        Rz = torch.stack([
            torch.stack([c, -s, zeros], dim=-1),
            torch.stack([s, c, zeros], dim=-1),
            torch.stack([zeros, zeros, ones], dim=-1)
        ], dim=-2)

        Xr_centered = torch.einsum('bic,acd->abid', coords_centered, Rz_t)
        Xr = Xr_centered + pl.unsqueeze(0).unsqueeze(2)
        Xr_flat = Xr.reshape(n_angles * B, 11, 3)

        if mean.dim() == 1: mean = mean.unsqueeze(0)
        if std.dim() == 1:  std  = std.unsqueeze(0)
        mean_tiled = mean.expand(n_angles * B, -1)
        std_tiled  = std.expand(n_angles * B, -1)

        ft, df, plt_, tht, _, _, _, Rt, spt, _, _ = self.get_features(Xr_flat, mean_tiled, std_tiled)
        out = self.forward(ft, df, plt_, tht, spt, Rt)
        pp  = out[0] if isinstance(out, tuple) else out

        pp   = pp.reshape(n_angles, B, 3)
        plt_ = plt_.reshape(n_angles, B, 3)
        pp_unrot = torch.einsum('abc,acd->abd', pp - plt_, Rz) + pl.unsqueeze(0)
        return pp_unrot.mean(dim=0)

print("유틸리티 함수 정의 완료")

유틸리티 함수 정의 완료


## 2. 모델 (model/architectures.py + model/architectures_calib.py)

In [3]:
class PriorBiasedLinear(nn.Module):
    """초기 출력이 prior_bias가 되도록 가중치/편향을 0으로 초기화한 Linear."""
    def __init__(self, in_features, out_features, prior_bias):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.register_buffer('prior_bias', prior_bias.clone().detach())
        with torch.no_grad():
            nn.init.zeros_(self.linear.weight)
            nn.init.zeros_(self.linear.bias)
    def forward(self, x):
        return self.linear(x) + self.prior_bias


class HyperPhysics_calib_xy2(BaseHyperPhysicsModel):
    """
    v_calib_xy2: v_xy2 + per-axis displacement calibration.
    dir_net(heading) + 4-term damping + 3-pair omega weighted avg
    + omega_net + rotation_gate + local NLL + Cartesian scale calib.
    """
    def __init__(self, input_dim=24, **kwargs):
        self.sh_thr    = kwargs.pop('sh_thr',    0.013012)
        self.sh_k      = kwargs.pop('sh_k',      408.348044)
        self.mse_w     = kwargs.pop('mse_w',     129.172037)
        self.local_w   = kwargs.pop('local_w',   0.050941)
        self.theta_thr = kwargs.pop('theta_thr', 1.087618)
        self.speed_thr = kwargs.pop('speed_thr', 0.034583)
        super().__init__(**kwargs)
        self.lr = 0.005400
        self.wd = 0.005659

        self.calib_log_scale = nn.Parameter(torch.zeros(3))
        self.calib_r_log     = nn.Parameter(torch.full((3,), -6.0))

        prior_dir = torch.tensor([-10.,-10.,-10.,-10.,-10.,-10.,-10.,0.,0.693,1.098])
        self.dir_net = nn.Sequential(
            nn.Linear(29, 24), nn.LayerNorm(24), nn.GELU(),
            PriorBiasedLinear(24, 10, prior_dir)
        )
        prior_ema = torch.zeros(6)
        self.temporal_net = nn.Sequential(
            nn.Linear(9, 32), nn.LayerNorm(32), nn.GELU(),
            PriorBiasedLinear(32, 6, prior_ema)
        )
        prior_dyn = torch.tensor([
            0.,0.,0.,0.,0.,0.,
            -4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,
            -4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,
        ])
        self.dynamics_net = nn.Sequential(
            nn.Linear(input_dim, 96), nn.LayerNorm(96), nn.GELU(),
            ResBlock(96),
            PriorBiasedLinear(96, 30, prior_dyn)
        )
        self.omega_w = nn.Parameter(torch.tensor([0.0, -0.5, -1.0]))
        self.omega_net = nn.Sequential(
            nn.LayerNorm(input_dim), nn.Linear(input_dim, 48), nn.GELU(), nn.Linear(48, 3)
        )
        with torch.no_grad():
            nn.init.normal_(self.omega_net[-1].weight, std=0.01)
            nn.init.zeros_(self.omega_net[-1].bias)
        self.diffusion_net = nn.Sequential(
            nn.Linear(input_dim, 32), nn.LayerNorm(32), nn.GELU(), nn.Linear(32, 3)
        )

    def _ema_va_local(self, diffs_local, alpha, beta):
        return _ema_va_local(diffs_local, alpha, beta)

    @staticmethod
    def _rotation_vector(d_prev, d_curr):
        n_prev = d_prev.norm(dim=1, keepdim=True).clamp(min=1e-8)
        n_curr = d_curr.norm(dim=1, keepdim=True).clamp(min=1e-8)
        d_hat_prev = d_prev / n_prev
        d_hat_curr = d_curr / n_curr
        cross  = torch.linalg.cross(d_hat_prev, d_hat_curr, dim=1)
        sin_t  = cross.norm(dim=1, keepdim=True).clamp(min=1e-8)
        cos_t  = (d_hat_prev * d_hat_curr).sum(1, keepdim=True).clamp(-0.9999, 0.9999)
        theta  = torch.atan2(sin_t, cos_t)
        speed_gate = torch.sigmoid((n_prev + n_curr) * 500 - 5)
        return cross / sin_t * theta * speed_gate

    def forward(self, features, diffs, p_last, theta, speed, R):
        dev = features.device
        B   = diffs.shape[0]

        ema_raw = self.temporal_net(features[:, 8:17])
        alpha = torch.sigmoid(ema_raw[:, 0:3]) * 0.8 + 0.1
        beta  = torch.sigmoid(ema_raw[:, 3:6]) * 0.199 + 0.8

        dyn_raw = self.dynamics_net(features)
        w_v = 2.0 + dyn_raw[:, 0:3]
        w_a = 1.0 + dyn_raw[:, 3:6]
        v_local_abs  = features[:, 17:20]
        v_local_abs2 = v_local_abs * v_local_abs
        theta2 = theta * theta

        exp_v = (F.softplus(dyn_raw[:,  6: 9]) * v_local_abs  +
                 F.softplus(dyn_raw[:,  9:12]) * v_local_abs2 +
                 F.softplus(dyn_raw[:, 12:15]) * theta        +
                 F.softplus(dyn_raw[:, 15:18]) * theta2)
        exp_a = (F.softplus(dyn_raw[:, 18:21]) * v_local_abs  +
                 F.softplus(dyn_raw[:, 21:24]) * v_local_abs2 +
                 F.softplus(dyn_raw[:, 24:27]) * theta        +
                 F.softplus(dyn_raw[:, 27:30]) * theta2)

        diffs_local = torch.matmul(diffs, R)
        vl, al = self._ema_va_local(diffs_local, alpha, beta)
        diff_speed = diffs_local.norm(dim=2)

        def rv_masked(ka, kb):
            rv    = self._rotation_vector(diffs_local[:, ka], diffs_local[:, kb])
            valid = ((diff_speed[:, ka] > 1e-5) & (diff_speed[:, kb] > 1e-5)).float()
            return rv * valid.unsqueeze(1), valid

        ov1, vm1 = rv_masked(-2, -1)
        ov2, vm2 = rv_masked(-3, -2)
        ov3, vm3 = rv_masked(-4, -3)

        w_logits = self.omega_w.view(1, 3).expand(B, -1)
        masks = torch.stack([vm1, vm2, vm3], dim=1)
        w_logits_masked = w_logits.masked_fill(masks == 0, -1e9)
        w_attn = F.softmax(w_logits_masked, dim=1)

        omega_hist = (w_attn[:, 0].unsqueeze(1) * ov1 +
                      w_attn[:, 1].unsqueeze(1) * ov2 +
                      w_attn[:, 2].unsqueeze(1) * ov3)

        current_speed = speed.view(B, 1) if speed is not None else diff_speed[:, -1].unsqueeze(1)
        omega_speed_gate = torch.sigmoid(current_speed * 500 - 5)
        omega_delta = self.omega_net(features) * omega_speed_gate

        theta_scalar = theta.view(B, 1)
        theta_gate = torch.sigmoid((theta_scalar - self.theta_thr) * 10)
        speed_gate_strong = torch.sigmoid((current_speed - self.speed_thr) * 200)
        rotation_gate = theta_gate * speed_gate_strong
        omega = (omega_hist + omega_delta) * rotation_gate

        v_rotated   = rodrigues_rotate(vl, omega)
        pred_local  = (w_v * torch.exp(-exp_v)) * v_rotated + (w_a * torch.exp(-exp_a)) * al
        pred_raw    = p_last + torch.einsum('nij,nj->ni', R, pred_local)

        r          = p_last.norm(dim=1, keepdim=True)
        scale_r    = 1.0 + torch.exp(self.calib_r_log.to(dev)) * r
        scale      = torch.exp(self.calib_log_scale.to(dev)) * scale_r
        delta      = pred_raw - p_last
        pred_global = p_last + delta * scale

        log_var = self.diffusion_net(features).clamp(min=-5.0, max=5.0)
        return pred_global, pred_local, log_var

    def compute_loss(self, pp, yr, pred_local=None, yr_local=None, log_var=None, p_last=None, theta=None, **kwargs):
        sh   = _soft_hit_loss(pp, yr, thr=self.sh_thr, k=self.sh_k)
        loss = sh + self.mse_w * F.mse_loss(pp, yr)
        if pred_local is not None and yr_local is not None and log_var is not None:
            sq_err   = (pred_local - yr_local) ** 2
            nll_loss = 0.5 * (torch.exp(-log_var) * sq_err + log_var)
            loss = loss + self.local_w * nll_loss.mean()
        elif pred_local is not None and yr_local is not None:
            loss = loss + self.local_w * F.mse_loss(pred_local, yr_local)
        return loss


# 파라미터 수 확인
m_tmp = HyperPhysics_calib_xy2()
total = sum(p.numel() for p in m_tmp.parameters())
print(f"HyperPhysics_calib_xy2 파라미터 수: {total:,}")
del m_tmp

HyperPhysics_calib_xy2 파라미터 수: 28,477


## 3. 데이터셋 (scripts/train/train.py)

In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class MosquitoPytorchDataset(Dataset):
    def __init__(self, X, y, device="cpu"):
        self.X = torch.tensor(X, dtype=torch.float32, device=device)
        self.y = torch.tensor(y, dtype=torch.float32, device=device)
    def __len__(self):  return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


class SlidingWindowDataset(Dataset):
    """
    슬라이딩 윈도우 데이터 증강.
    mode='extended': target_idx ∈ {4,5,6,7,8,9,10,12} → ~37배 증강
    pad_mode='acc': 등가속도 외삽 패딩
    """
    def __init__(self, X, y, min_win=3, mode="extended", device="cpu", pad_mode="acc"):
        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_tensor = torch.tensor(y, dtype=torch.float32)

        windows = []
        if mode == "extended":
            targets = [4, 5, 6, 7, 8, 9, 10, 12]
        elif mode == "1step":
            targets = [3, 4, 5, 6, 7, 8, 9, 10]
        else:
            targets = [12, 10]

        for i in range(len(X)):
            for target_idx in targets:
                S = 2
                end_idx = target_idx - S
                max_w   = end_idx + 2 if mode == "extended" else (12 if target_idx == 12 else 10)
                for w in range(min_win, max_w):
                    windows.append((i, w, target_idx, S))

        X_list, y_list = [], []
        for i, w, target_idx, S in windows:
            X_orig  = X_tensor[i]
            end_idx = target_idx - S
            pts     = X_orig[end_idx - w + 1 : end_idx + 1]
            target  = y_tensor[i] if target_idx == 12 else X_orig[target_idx]

            if w < 11:
                n_pad = 11 - w
                if pad_mode == "vel":
                    v0  = pts[1] - pts[0]
                    pad = torch.stack([pts[0] - (n_pad - k) * v0 for k in range(n_pad)])
                elif pad_mode == "acc" and w >= 3:
                    v0  = pts[1] - pts[0]
                    a0  = pts[2] - 2 * pts[1] + pts[0]
                    pad = torch.stack([pts[0] - j*v0 + j*(j-1)/2*a0
                                       for j in range(n_pad, 0, -1)])
                else:
                    pad = pts[0:1].expand(n_pad, -1)
                X_padded = torch.cat([pad, pts], dim=0)
            else:
                X_padded = pts.clone()

            X_list.append(X_padded)
            y_list.append(target)

        self.X_all = torch.stack(X_list).to(device)
        self.y_all = torch.stack(y_list).to(device)
        print(f"SlidingWindowDataset: {len(X)}개 → {len(self.X_all)}개 ({len(self.X_all)/len(X):.1f}×)")

    def __len__(self):  return len(self.X_all)
    def __getitem__(self, idx): return self.X_all[idx], self.y_all[idx]

print("데이터셋 클래스 정의 완료")

데이터셋 클래스 정의 완료


## 4. 데이터 로드

In [5]:
X_all = np.load(f"{CACHE_DIR}/train_coords.npy")
y_all = np.load(f"{CACHE_DIR}/train_targets.npy")
print(f"X_all: {X_all.shape}  y_all: {y_all.shape}")

X_all: (10000, 11, 3)  y_all: (10000, 3)


## 5. 5-Fold 학습

`train_parallel.py -v v_calib_xy2 --seed 42` 와 동일한 로직 (순차 실행).

In [6]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
fold_scores = []

for fold, (tri, vli) in enumerate(kf.split(X_all)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold+1}/5")
    print(f"{'='*60}")

    set_seed(SEED)  # 병렬 실행과 동일하게 각 fold를 같은 seed로 초기화

    Xtr, Xvl = X_all[tri], X_all[vli]
    ytr, yvl = y_all[tri], y_all[vli]

    train_ds = SlidingWindowDataset(Xtr, ytr, min_win=MIN_WIN,
                                    mode=SLIDING_MODE, device=device, pad_mode=PAD_MODE)
    trl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    vll = DataLoader(MosquitoPytorchDataset(Xvl, yvl, device=device),
                     batch_size=256, shuffle=False)

    model = HyperPhysics_calib_xy2().to(device)
    lr = getattr(model, 'lr', 1e-2)
    wd = getattr(model, 'wd', 3e-3)

    param_groups = model.get_param_groups(lr, wd)
    opt   = torch.optim.AdamW(param_groups)
    sched = StepLR(opt, step_size=LR_DECAY_STEP, gamma=LR_DECAY_GAMMA)

    # 정규화 통계 계산 (학습 데이터 기준)
    with torch.no_grad():
        _, _, _, _, _, _, _, _, _, mn, st = model.get_features(
            torch.tensor(Xtr, dtype=torch.float32, device=device)
        )

    best = 0.0
    for ep in range(1, EPOCHS + 1):
        model.train()
        tloss = 0.0
        for Xb, yb in trl:
            B   = Xb.shape[0]
            pb  = Xb[:, 10]
            opt.zero_grad(set_to_none=True)
            ft, df, plt_, tht, _, _, _, Rt, spt, _, _ = model.get_features(Xb, mn, st)
            out = model(ft, df, plt_, tht, spt, Rt)
            pp, pred_local, log_var = out
            yr_local = torch.matmul((yb - plt_).unsqueeze(1), Rt).squeeze(1)
            loss = model.compute_loss(pp, yb, pred_local, yr_local, log_var, p_last=plt_, theta=tht)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            tloss += loss.item() * B
        sched.step()
        tloss /= len(trl.dataset)

        model.eval()
        hits = 0
        with torch.no_grad():
            for Xv, yv in vll:
                pv   = model.predict_batch_tta(Xv, mn, st, n_angles=1)
                hits += (torch.norm(pv - yv, dim=1) <= 0.01).sum().item()
        hr = hits / len(Xvl)

        if hr > best:
            best = hr
            torch.save(model.state_dict(),
                       f"{CKPT_DIR}/best_{VERSION}_seed{SEED}_fold{fold+1}.pth")
            torch.save({"mean": mn, "std": st},
                       f"{CKPT_DIR}/{VERSION}_seed{SEED}_stats_fold{fold+1}.pt")

        print(f"  Ep {ep:2d}/{EPOCHS} | Loss {tloss:.4f} | Val {hr*100:.3f}% (Best {best*100:.3f}%)")

    print(f"FOLD {fold+1} Best: {best*100:.3f}%")
    fold_scores.append(best)

print(f"\n{'='*60}")
print(f"5-Fold CV: {np.mean(fold_scores)*100:.3f}%")
print(f"{'='*60}")


FOLD 1/5
SlidingWindowDataset: 8000개 → 296000개 (37.0×)
  Ep  1/50 | Loss 0.2670 | Val 67.200% (Best 67.200%)
  Ep  2/50 | Loss 0.2511 | Val 66.400% (Best 67.200%)
  Ep  3/50 | Loss 0.2491 | Val 67.100% (Best 67.200%)
  Ep  4/50 | Loss 0.2477 | Val 67.850% (Best 67.850%)
  Ep  5/50 | Loss 0.2466 | Val 67.500% (Best 67.850%)
  Ep  6/50 | Loss 0.2457 | Val 67.200% (Best 67.850%)
  Ep  7/50 | Loss 0.2450 | Val 67.850% (Best 67.850%)
  Ep  8/50 | Loss 0.2444 | Val 67.600% (Best 67.850%)
  Ep  9/50 | Loss 0.2421 | Val 67.900% (Best 67.900%)
  Ep 10/50 | Loss 0.2411 | Val 67.950% (Best 67.950%)
  Ep 11/50 | Loss 0.2406 | Val 67.750% (Best 67.950%)
  Ep 12/50 | Loss 0.2402 | Val 67.200% (Best 67.950%)
  Ep 13/50 | Loss 0.2395 | Val 68.000% (Best 68.000%)
  Ep 14/50 | Loss 0.2391 | Val 67.500% (Best 68.000%)
  Ep 15/50 | Loss 0.2386 | Val 68.050% (Best 68.050%)
  Ep 16/50 | Loss 0.2382 | Val 67.550% (Best 68.050%)
  Ep 17/50 | Loss 0.2364 | Val 67.350% (Best 68.050%)
  Ep 18/50 | Loss 0.2358 |

## 6. 체크포인트 로드

In [7]:
fold_models = []  # [(model, mean, std), ...] × 5
for fold in range(1, 6):
    pth     = f"{CKPT_DIR}/best_{VERSION}_seed{SEED}_fold{fold}.pth"
    stats_p = f"{CKPT_DIR}/{VERSION}_seed{SEED}_stats_fold{fold}.pt"
    m = HyperPhysics_calib_xy2().to(device)
    m.load_state_dict(torch.load(pth, map_location=device))
    m.eval()
    s = torch.load(stats_p, map_location=device)
    fold_models.append((m, s["mean"].to(device), s["std"].to(device)))
    print(f"Fold {fold} loaded")

print("체크포인트 로드 완료")

Fold 1 loaded
Fold 2 loaded
Fold 3 loaded
Fold 4 loaded
Fold 5 loaded
체크포인트 로드 완료


## 7. OOF 슬라이딩 윈도우 예측 수집

In [8]:
def _pad_window_torch(pts, n_pad, pad_mode, dev):
    """pts: (B, w, 3) → (B, 11, 3) acc 패딩"""
    if n_pad == 0:
        return pts
    v0 = pts[:, 1] - pts[:, 0]
    if pad_mode == "vel":
        js  = torch.arange(n_pad, 0, -1, device=dev).float()
        pad = pts[:, 0:1] - js.view(1, n_pad, 1) * v0.unsqueeze(1)
    elif pad_mode == "acc" and pts.shape[1] >= 3:
        a0  = pts[:, 2] - 2*pts[:, 1] + pts[:, 0]
        js  = torch.arange(n_pad, 0, -1, device=dev).float()
        pad = (pts[:, 0:1]
               - js.view(1, n_pad, 1) * v0.unsqueeze(1)
               + (js*(js-1)/2).view(1, n_pad, 1) * a0.unsqueeze(1))
    else:
        pad = pts[:, 0:1].expand(-1, n_pad, -1)
    return torch.cat([pad, pts], dim=1)


WINDOWS  = list(range(3, 12))  # [3,4,...,11] → 9개

# oof_win_preds[seed_idx, sample_idx, win_idx, coord] — seed 1개만 사용
oof_win_preds = np.zeros((1, len(X_all), len(WINDOWS), 3), dtype=np.float32)

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
sample_to_fold = np.zeros(len(X_all), dtype=np.int32)
for fold_idx, (_, vli) in enumerate(kf.split(X_all)):
    sample_to_fold[vli] = fold_idx

print("OOF 슬라이딩 윈도우 예측 수집 중...")
for fold_idx in range(5):
    vli   = np.where(sample_to_fold == fold_idx)[0]
    X_val = torch.tensor(X_all[vli], dtype=torch.float32, device=device)
    m, mn, st = fold_models[fold_idx]
    with torch.no_grad():
        for w_idx, w in enumerate(WINDOWS):
            preds = []
            for s in range(0, len(X_val), INFER_BATCH):
                batch = X_val[s:s+INFER_BATCH]
                b = _pad_window_torch(batch[:, 11-w:], 11-w, PAD_MODE, device)
                preds.append(m.predict_batch_tta(b, mn, st, n_angles=1).cpu().numpy())
            oof_win_preds[0, vli, w_idx] = np.concatenate(preds)
    print(f"  Fold {fold_idx+1} done")

print(f"oof_win_preds shape: {oof_win_preds.shape}")

OOF 슬라이딩 윈도우 예측 수집 중...
  Fold 1 done
  Fold 2 done
  Fold 3 done
  Fold 4 done
  Fold 5 done
oof_win_preds shape: (1, 10000, 9, 3)


## 8. Optuna 윈도우 가중치 최적화

`predict_optimized.py --trials 1000` 와 동일.

In [12]:
def evaluate_weights(win_weights, seed_oof, y_tr):
    ww = win_weights / (win_weights.sum() + 1e-8)
    pred = np.einsum("nwc,w->nc", seed_oof, ww)
    return float((np.linalg.norm(pred - y_tr, axis=1) <= 0.01).mean())


optuna.logging.set_verbosity(optuna.logging.WARNING)
seed_oof = oof_win_preds[0]  # (N, 9, 3)

def objective(trial):
    win_w = np.array([trial.suggest_float(f"w_win_{i}", 0.0, 1.0)
                      for i in range(3, 12)], dtype=np.float32)
    return evaluate_weights(win_w, seed_oof, y_all)

print(f"Optuna 탐색 중 ({TRIALS} trials)...")
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study.optimize(objective, n_trials=TRIALS, show_progress_bar=True)

best_win = np.array([study.best_params[f"w_win_{i}"] for i in range(3, 12)], dtype=np.float32)
best_win /= best_win.sum() + 1e-8

oof_hr = evaluate_weights(best_win, seed_oof, y_all)
print(f"\nOptuna Best OOF HR: {oof_hr*100:.4f}%")
print(f"Window weights: {np.round(best_win, 4).tolist()}")

Optuna 탐색 중 (1000 trials)...


  0%|          | 0/1000 [00:00<?, ?it/s]


Optuna Best OOF HR: 67.9600%
Window weights: [0.03060000017285347, 0.04450000077486038, 0.049800001084804535, 0.20999999344348907, 0.24420000612735748, 0.04259999841451645, 0.19220000505447388, 0.0706000030040741, 0.11559999734163284]


## 9. 테스트 추론 + Submission 저장

In [10]:
# 테스트 데이터 로드
df_sub     = pd.read_csv(SAMPLE_SUB)
all_coords = [
    pd.read_csv(os.path.join(TEST_DIR, f"{row['id']}.csv"))[["x","y","z"]].values
    for _, row in df_sub.iterrows()
]
test_coords = np.array(all_coords, dtype=np.float32)
print(f"테스트셋: {test_coords.shape}")

N = len(test_coords)
all_preds  = np.zeros((N, 3), dtype=np.float32)
best_fold  = np.ones(5, dtype=np.float32) / 5.0  # 균등 폴드 가중치

print(f"추론 중 (TTA angles={TTA_ANGLES})...")
with torch.no_grad():
    for start in range(0, N, INFER_BATCH):
        end   = min(start + INFER_BATCH, N)
        batch = torch.tensor(test_coords[start:end], dtype=torch.float32, device=device)

        batch_pred = torch.zeros(end - start, 3, device=device)
        for fold_idx, (m, mn, st) in enumerate(fold_models):
            fold_pred = torch.zeros(end - start, 3, device=device)
            for w_idx, w in enumerate(WINDOWS):
                b = _pad_window_torch(batch[:, 11-w:], 11-w, PAD_MODE, device)
                p = m.predict_batch_tta(b, mn, st, n_angles=TTA_ANGLES, use_mirrors=False)
                fold_pred += p * float(best_win[w_idx])
            batch_pred += fold_pred * float(best_fold[fold_idx])

        all_preds[start:end] = batch_pred.cpu().numpy()
        if end % 2000 == 0 or end == N:
            print(f"  {end:5d} / {N}")

# 저장
acc_str    = f"{oof_hr*100:.4f}"
out_path   = f"data/submission_{VERSION}_opt_tta_{acc_str}.csv"
info_path  = f"data/submission_{VERSION}_opt_tta_{acc_str}.info"

df_sub[["x","y","z"]] = all_preds
df_sub.to_csv(out_path, index=False)

info = (
    f"Model: {VERSION} (HyperPhysics_calib_xy2)\n"
    f"Seed: {SEED}\n"
    f"Window Weights (w=3..11): {np.round(best_win, 6).tolist()}\n"
    f"OOF HR: {oof_hr*100:.4f}%\n"
)
with open(info_path, "w") as f:
    f.write(info)

# 표준 경로에도 복사
df_sub.to_csv("data/submission.csv", index=False)
with open("data/submission.info", "w") as f:
    f.write(info)

print(f"\nSubmission saved: {out_path}")
print(f"Info saved:       {info_path}")
print(f"Standard copy:    data/submission.csv")

테스트셋: (10000, 11, 3)
추론 중 (TTA angles=1)...
  10000 / 10000

Submission saved: data/submission_v_calib_xy2_opt_tta_67.9600.csv
Info saved:       data/submission_v_calib_xy2_opt_tta_67.9600.info
Standard copy:    data/submission.csv


## 10. 결과 요약

In [11]:
print("=" * 60)
print(f"모델: {VERSION}")
print(f"Seed: {SEED}")
print()
for i, sc in enumerate(fold_scores):
    print(f"  Fold {i+1}: {sc*100:.3f}%")
print(f"  Avg CV: {np.mean(fold_scores)*100:.3f}%")
print()
print(f"Optuna OOF HR ({TRIALS} trials): {oof_hr*100:.4f}%")
print(f"Submission: {out_path}")
print("=" * 60)

모델: v_calib_xy2
Seed: 42

  Fold 1: 68.050%
  Fold 2: 67.850%
  Fold 3: 68.600%
  Fold 4: 67.450%
  Fold 5: 67.650%
  Avg CV: 67.920%

Optuna OOF HR (1000 trials): 67.9600%
Submission: data/submission_v_calib_xy2_opt_tta_67.9600.csv
